In [1]:
from pymongo import MongoClient
from getpass import getpass

# Verbindung aufbauen
#password = getpass("Bitte MongoDB Passwort eingeben (falls nötig): ")
client = MongoClient(f"mongodb://localhost:27017/") # Ohne Passwort: MongoClient("mongodb://localhost:27017/")
db = client['Espana'] # Dein Datenbankname
collection = db['ciudad']

# Die Aggregation-Pipeline
pipeline = [
    # 1. Prüfen, ob das Feld existiert und nicht leer ist
    {
        "$match": {
            "water": { "$exists": True, "$ne": "" }
        }
    },
    # 2. Das Objekt in ein Array aus Key-Value Paaren umwandeln
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "water_values": { "$objectToArray": "$water" }
        }
    },
    # 3. Berechnen des Durchschnitts pro Stadt (Dokument)
    {
        "$addFields": {
            "average_water": { "$avg": "$water_values.v" }
        }
    },

    # 4. Min-Temperatur pro Stadt (Dokument)

    {
        "$addFields": {
            "min_water": { "$min": "$water_values.v" }
        }
    },

        {
        "$addFields": {
            "max_water": { "$max": "$water_values.v" }
        }
    },
    
    # 4. Sortieren nach dem höchsten Durchschnitt
    {
        "$sort": { "average_water": -1 }
    },

    
]



# Abfrage ausführen
results = collection.aggregate(pipeline)

print(f"{'Ciudad':<30} | {'Provincia':<25} | {'Avg Water':<10} | {'Min Water':<10} | {'Max Water':<10} | {'Differenz':<10}"  )
print("-" * 110)

for res in results:
    # Beide Werte direkt aus dem aktuellen Dokument ziehen
    avg = round(res.get('average_water', 0), 2)
    min_val = round(res.get('min_water', 0), 2)
    max_val = round(res.get('max_water', 0), 2)

    diff_val = max_val - min_val
    
    print(f"{res['Ciudad']:<30} | {res['Provincia']:<25} | {avg:<10} | {min_val:<10} | {max_val:<10} | {diff_val:<10} ")

Ciudad                         | Provincia                 | Avg Water  | Min Water  | Max Water  | Differenz 
--------------------------------------------------------------------------------------------------------------
Puntallana                     | Santa Cruz de Tenerife    | 21.25      | 19         | 24         | 5          
Los Llanos de Aridane          | Santa Cruz de Tenerife    | 21.25      | 19         | 24         | 5          
Santa Cruz de La Palma         | Santa Cruz de Tenerife    | 21.25      | 19         | 24         | 5          
La Oliva                       | Las Palmas                | 20.42      | 18         | 23         | 5          
Pájara                         | Las Palmas                | 20.42      | 18         | 23         | 5          
Puerto del Rosario             | Las Palmas                | 20.42      | 18         | 23         | 5          
Teguise                        | Las Palmas                | 20.25      | 18         | 23         | 5     

In [2]:
# Verbindung aufbauen
client = MongoClient("mongodb://localhost:27017/") 
db = client['Espana'] 
collection = db['ciudad']

# Die optimierte Aggregation-Pipeline
pipeline = [
    # 1. Prüfen, ob das Feld existiert und nicht leer ist
    {
        "$match": {
            "water": { "$exists": True, "$ne": "" }
        }
    },
    # 2. Das Objekt in ein Array aus Key-Value Paaren umwandeln
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "water_values": { "$objectToArray": "$water" }
        }
    },
    # 3. Berechnen von Durchschnitt, Min und Max
    {
        "$addFields": {
            "average_water": { "$avg": "$water_values.v" },
            "min_water": { "$min": "$water_values.v" },
            "max_water": { "$max": "$water_values.v" }
        }
    },
    # 4. Berechnung der Differenz innerhalb der Pipeline
    {
        "$addFields": {
            "difference_water": { "$subtract": ["$max_water", "$min_water"] }
        }
    },
    # 5. Filter: Nur Orte mit einer Differenz von maximal 5 Grad
    {
        "$match": {
            "difference_water": { "$lte": 5 }
        }
    },
    # 6. Sortieren nach dem höchsten Durchschnitt
    {
        "$sort": { "average_water": -1 }
    }
]

# Abfrage ausführen
results = collection.aggregate(pipeline)

print(f"{'Ciudad':<30} | {'Provincia':<25} | {'Avg Water':<10} | {'Min Water':<10} | {'Max Water':<10} | {'Differenz':<10}")
print("-" * 110)

for res in results:
    avg = round(res.get('average_water', 0), 2)
    min_val = round(res.get('min_water', 0), 2)
    max_val = round(res.get('max_water', 0), 2)
    diff = round(res.get('difference_water', 0), 2)
    
    print(f"{res['Ciudad']:<30} | {res['Provincia']:<25} | {avg:<10} | {min_val:<10} | {max_val:<10} | {diff:<10}")

Ciudad                         | Provincia                 | Avg Water  | Min Water  | Max Water  | Differenz 
--------------------------------------------------------------------------------------------------------------
Puntallana                     | Santa Cruz de Tenerife    | 21.25      | 19         | 24         | 5         
Los Llanos de Aridane          | Santa Cruz de Tenerife    | 21.25      | 19         | 24         | 5         
Santa Cruz de La Palma         | Santa Cruz de Tenerife    | 21.25      | 19         | 24         | 5         
La Oliva                       | Las Palmas                | 20.42      | 18         | 23         | 5         
Pájara                         | Las Palmas                | 20.42      | 18         | 23         | 5         
Puerto del Rosario             | Las Palmas                | 20.42      | 18         | 23         | 5         
Teguise                        | Las Palmas                | 20.25      | 18         | 23         | 5         
H